## <로스엔젤레스 박물관 데이터 수집>
- 스크래핑을 통한 수집 (JSON API)

### 1. 필요한 것들 불러오기

In [ ]:
import os
import time
import requests
import pandas as pd

MUSEUM_CODE = "LACMA"
OUTPUT_EXCEL = f"../data/{MUSEUM_CODE}_hyungbae.xlsx"
HEADERS = {
    "User-Agent": "aks-digital-humanities-research/1.0",
    "Content-Type": "application/json",
}

### 2. API 호출해서 원하는 정보 얻기

In [ ]:
CATEGORY = "흉배"
CATEGORY_LETTER = "H"
KEYWORD = "rank badge"

rows = []
serial_counter = {}

def next_temp_id(museum_code):
    n = serial_counter.get(museum_code, 0) + 1
    serial_counter[museum_code] = n
    return f"Y{museum_code}{CATEGORY_LETTER}{n:02d}"

def fetch_lacma(keyword=KEYWORD, per_page=48):
    url = "https://collections.lacma.org/api/search"
    page = 1
    all_items = []

    while True:
        payload = {
            "query": keyword,
            "classification": [], "department": [], "artist": [], "placeMade": [],
            "creditLine": [], "culture": [], "period": [], "style": [],
            "building": [], "gallery": [],
            "onView": False, "hasImage": True, "publicDomain": False,
            "sort": "RELEVANCE", "page": page, "perPage": per_page,
        }
        resp = requests.post(url, headers=HEADERS, json=payload, timeout=20)
        resp.raise_for_status()
        data = resp.json()
        results = data.get("results") or []
        all_items.extend(results)

        total = data.get("total", 0)
        if page * per_page >= total or not results:
            break
        page += 1
        time.sleep(0.3)

    print(f"전체 검색 결과: {len(all_items)}건 (필터링 전)")

    for item in all_items:
        obj = (item.get("data") or {}).get("object") or {}

        titles = obj.get("titles") or []
        # 영어명: Primary Title 중 displayOrder가 가장 앞선 것
        primary_titles = [t for t in titles if t.get("titleType") == "Primary Title"]
        primary_titles.sort(key=lambda t: t.get("displayOrder", 999))
        title_en = primary_titles[0]["title"] if primary_titles else (titles[0]["title"] if titles else None)

        # 한글명: 번역 제목이 있으면 사용
        ko_titles = [t for t in titles if "translation" in (t.get("titleType") or "").lower()]
        title_ko = ko_titles[0]["title"] if ko_titles else ""

        classification = obj.get("classification")
        # Costumes 분류가 아니면 흉배가 아닐 가능성이 높음 -> 필터링
        if classification != "Costumes":
            continue
        # 제목에 흉배 관련 단어가 없으면 제외
        title_check = (title_en or "").lower()
        if not any(w in title_check for w in ["badge", "buzi", "hyungbae"]):
            continue

        temp_id = next_temp_id(MUSEUM_CODE)

        images = obj.get("images") or []
        image_urls = [img.get("renditions", {}).get("desktop") for img in images if img.get("renditions", {}).get("desktop")]

        constituents = obj.get("constituents") or []
        artists = "; ".join(c.get("displayName", "") for c in constituents if c.get("displayName"))

        place_made = obj.get("placeMade") or []
        place_str = "; ".join(place_made) if isinstance(place_made, list) else place_made

        rows.append({
            "임시ID": temp_id,
            "분류": CATEGORY,
            "소장처": "로스앤젤레스 카운티 미술관(LACMA)",
            "소장처유물번호": obj.get("accessionNumber"),
            "한글명": title_ko,
            "한자명": "",
            "영어명": title_en,
            "URL": f"https://collections.lacma.org/object/{item.get('id')}",
            "searched_keyword": keyword,
            "classification": classification,
            "department": obj.get("department"),
            "artist": artists,
            "date": obj.get("dated"),
            "medium": obj.get("medium"),
            "dimensions": obj.get("dimensions"),
            "credit_line": obj.get("creditLine"),
            "place_made": place_str,
            "culture": obj.get("culture"),
            "image_url": image_urls[0] if image_urls else None,
            "image_urls": "; ".join(image_urls),
            "license_note": "© Museum Associates/LACMA (건별 라이선스 확인 필요)",
        })

fetch_lacma()
print(f"필터링 후 흉배 대상: {len(rows)}건")

objectID 72339 실패: 404 Client Error: Not Found for url: https://collectionapi.metmuseum.org/public/collection/v1/objects/72339
총 1건 실패, 나중에 재시도 필요: [72339]
257건 수집 완료


### 3. 데이터 프레임 -> 엑셀

In [ ]:
df = pd.DataFrame(rows)
master_cols = ["임시ID", "분류", "소장처", "소장처유물번호", "한글명", "한자명", "영어명",
               "URL", "searched_keyword", "classification", "department", "artist", "date",
               "medium", "dimensions", "credit_line", "place_made", "culture",
               "image_url", "image_urls", "license_note"]
other_cols = [c for c in df.columns if c not in master_cols]
df = df[master_cols + other_cols]

os.makedirs("../data", exist_ok=True)
df.to_excel(OUTPUT_EXCEL, index=False)
print(df.shape)
df.head()

(209, 21)


,임시ID,분류,소장처,소장처유물번호,한글명,한자명,영어명,URL,culture,period,...,medium,classification,dimensions,credit_line,curatorial_department,image_urls,is_public_domain,license_note,searched_keyword,image_url
0,YMETH01,흉배,메트로폴리탄 미술관,Met 30.75.968,,,Rank Badge,https://www.metmuseum.org/art/collection/searc...,China,Qing dynasty (1644–1911),...,Silk on silk,Textiles-Embroidered,Overall: 12 1/4 x 12 1/2 in. (31.1 x 31.8cm),"Bequest of William Christian Paul, 1929",Asian Art,https://images.metmuseum.org/CRDImages/as/orig...,True,Met Open Access (CC0 for public-domain works),rank badge,https://images.metmuseum.org/CRDImages/as/orig...
1,YMETH02,흉배,메트로폴리탄 미술관,Met 53.60.20,,,Rank Badge,https://www.metmuseum.org/art/collection/searc...,Korea,Joseon dynasty (1392–1910),...,Silk,Textiles-Embroidered,7 3/4 x 6 3/4 in. (19.7 x 17.1 cm),"Seymour Fund, 1953",Asian Art,https://images.metmuseum.org/CRDImages/as/orig...,True,Met Open Access (CC0 for public-domain works),rank badge,https://images.metmuseum.org/CRDImages/as/orig...
2,YMETH03,흉배,메트로폴리탄 미술관,Met 53.60.21,,,Rank Badge,https://www.metmuseum.org/art/collection/searc...,Korea,Joseon dynasty (1392–1910),...,Silk,Textiles-Embroidered,7 1/2 x 6 1/2 in. (19.1 x 16.5 cm),"Seymour Fund, 1953",Asian Art,https://images.metmuseum.org/CRDImages/as/orig...,True,Met Open Access (CC0 for public-domain works),rank badge,https://images.metmuseum.org/CRDImages/as/orig...
3,YMETH04,흉배,메트로폴리탄 미술관,Met 53.60.22,,,Rank Badge,https://www.metmuseum.org/art/collection/searc...,Korea,Joseon dynasty (1392–1910),...,Silk,Textiles-Embroidered,7 1/2 x 6 1/2 in. (19.1 x 16.5 cm),"Seymour Fund, 1953",Asian Art,https://images.metmuseum.org/CRDImages/as/orig...,True,Met Open Access (CC0 for public-domain works),rank badge,https://images.metmuseum.org/CRDImages/as/orig...
4,YMETH05,흉배,메트로폴리탄 미술관,Met 53.60.16,,,Rank Badge,https://www.metmuseum.org/art/collection/searc...,Korea,Joseon dynasty (1392–1910),...,Silk,Textiles-Embroidered,8 1/2 x 7 3/4 in. (21.6 x 19.7 cm),"Seymour Fund, 1953",Asian Art,https://images.metmuseum.org/CRDImages/as/orig...,True,Met Open Access (CC0 for public-domain works),rank badge,https://images.metmuseum.org/CRDImages/as/orig...


### 4. 이미지

In [20]:
# 이미지 저장 폴더 (NFM 노트북이랑 동일한 경로 구조)
os.makedirs("../image/hyungbae", exist_ok=True)

pure_hyungbae = pure_hyungbae.reset_index(drop=True)

for i, row in pure_hyungbae.iterrows():
    temp_id = row["임시ID"]
    raw = row["image_urls"]

    if pd.isna(raw) or str(raw).strip() == "":
        continue  # 이미지 없는 유물은 건너뜀

    image_urls = [u for u in str(raw).split("; ") if u.strip()]

    for idx, image_url in enumerate(image_urls):
        suffix = "" if len(image_urls) == 1 else f"-{idx + 1}"
        filepath = f"../image/hyungbae/{temp_id}{suffix}.jpg"

        resp = requests.get(image_url, headers=HEADERS, timeout=30)
        with open(filepath, "wb") as f:
            f.write(resp.content)

    if (i + 1) % 20 == 0:
        print(f"{i + 1} / {len(pure_hyungbae)} 완료")

    time.sleep(0.5)

20 / 209 완료
40 / 209 완료
60 / 209 완료
80 / 209 완료
100 / 209 완료
120 / 209 완료
140 / 209 완료
160 / 209 완료
180 / 209 완료
